### 1. Importing Libraries and Loading the Dataset

In [1]:

import json

# Load the JSON dataset
with open("benchmark.json", "r", encoding="utf-8") as file:
    data = json.load(file)

# Inspect the dataset
print("Data type:", type(data))

print("\nDataset preview:")
print(json.dumps(data[:2] if isinstance(data, list) else data, indent=4)[:3000])


Data type: <class 'list'>

Dataset preview:
[
    {
        "id": "hf-00000",
        "category": "single_ingredient",
        "ocr_text": "Avastin 400mg Injection\nEach injection contains:\nBevacizumab 400mg\nRoche Products India Pvt Ltd",
        "expected_names": [
            "Bevacizumab"
        ],
        "source_composition": "Bevacizumab (400mg)"
    },
    {
        "id": "hf-00001",
        "category": "dual_ingredient",
        "ocr_text": "Augmentin 625 Duo Tablet\nEach tablet contains:\nAmoxycillin 500mg\nClavulanic Acid 125mg\nGlaxo SmithKline Pharmaceuticals Ltd",
        "expected_names": [
            "Amoxycillin",
            "Clavulanic Acid"
        ],
        "source_composition": "Amoxycillin  (500mg) +  Clavulanic Acid (125mg)"
    }
]


### 2. Exploring the Medicine Dataset

In [2]:

import pandas as pd

# Convert JSON data into a DataFrame
df = pd.DataFrame(data)

# Display dataset information
print("Total records:", len(df))

print("\nAvailable columns:")
print(df.columns.tolist())

print("\nMedicine categories:")
print(df["category"].value_counts())

print("\nFirst 5 records:")
print(df[["id", "category", "ocr_text", "expected_names"]].head())


Total records: 500

Available columns:
['id', 'category', 'ocr_text', 'expected_names', 'source_composition']

Medicine categories:
category
single_ingredient    279
dual_ingredient      155
multi_ingredient      66
Name: count, dtype: int64

First 5 records:
         id           category  \
0  hf-00000  single_ingredient   
1  hf-00001    dual_ingredient   
2  hf-00002  single_ingredient   
3  hf-00003   multi_ingredient   
4  hf-00004  single_ingredient   

                                            ocr_text  \
0  Avastin 400mg Injection\nEach injection contai...   
1  Augmentin 625 Duo Tablet\nEach tablet contains...   
2  Azithral 500 Tablet\nAzithromycin (500mg)\nAle...   
3  Ascoril LS Syrup\nComposition: Ambroxol (30mg/...   
4  Aciloc 150 Tablet\nComposition: Ranitidine (15...   

                            expected_names  
0                            [Bevacizumab]  
1           [Amoxycillin, Clavulanic Acid]  
2                           [Azithromycin]  
3  [Ambroxol, Levo

### 3. Initial Ingredient Extraction Using Regex

In [6]:

import re

# Step 1: Create a list of all known ingredient names
ingredient_list = set()

for names in df["expected_names"]:
    for name in names:
        ingredient_list.add(name)

ingredient_list = sorted(ingredient_list, key=len, reverse=True)

print("Total unique ingredients:", len(ingredient_list))
print("\nSample ingredients:")
print(ingredient_list[:20])


# Step 2: Create ingredient extraction function

def extract_ingredients(ocr_text):

    matches = []

    # Find all ingredient matches with their positions
    for ingredient in ingredient_list:

        pattern = (
            r'(?<!\w)' +
            re.escape(ingredient) +
            r'(?!\w)'
        )

        for match in re.finditer(pattern, ocr_text, re.IGNORECASE):
            matches.append({
                "name": ingredient,
                "start": match.start(),
                "end": match.end()
            })

    # Sort by length, longest ingredient first
    matches.sort(
        key=lambda x: x["end"] - x["start"],
        reverse=True
    )

    selected = []

    # Remove overlapping matches
    for match in matches:

        overlapping = any(
            match["start"] < item["end"] and
            match["end"] > item["start"]
            for item in selected
        )

        if not overlapping:
            selected.append(match)

    return list(dict.fromkeys(
        item["name"] for item in selected
    ))



# Step 3: Test on the first five records
for _, row in df.head(5).iterrows():

    detected = extract_ingredients(row["ocr_text"])

    print("\nMedicine ID:", row["id"])
    print("OCR Text:", row["ocr_text"])
    print("Expected Ingredients:", row["expected_names"])
    print("Detected Ingredients:", detected)


Total unique ingredients: 199

Sample ingredients:
['Recombinant Human Erythropoietin Alfa', 'Dextromethorphan Hydrobromide', 'Hydroxypropylmethylcellulose', 'Disodium Hydrogen Citrate', 'Chlorpheniramine Maleate', 'L-Ornithine L-Aspartate', 'L-Methyl Folate Calcium', 'Carboxymethylcellulose', 'Fluticasone Propionate', 'Sodium acid phosphate', 'Oseltamivir Phosphate', 'Ursodeoxycholic Acid', 'Stabilized Oxychloro', 'Azilsartan medoxomil', 'Metoprolol Succinate', 'Fluticasone Furoate', 'Choline fenofibrate', 'Hydrochlorothiazide', 'Aluminium Hydroxide', 'Hydroxyprogesterone']

Medicine ID: hf-00000
OCR Text: Avastin 400mg Injection
Each injection contains:
Bevacizumab 400mg
Roche Products India Pvt Ltd
Expected Ingredients: ['Bevacizumab']
Detected Ingredients: ['Bevacizumab']

Medicine ID: hf-00001
OCR Text: Augmentin 625 Duo Tablet
Each tablet contains:
Amoxycillin 500mg
Clavulanic Acid 125mg
Glaxo SmithKline Pharmaceuticals Ltd
Expected Ingredients: ['Amoxycillin', 'Clavulanic Acid']

### 4. Evaluating Ingredient Extraction Performance

In [9]:

# Step 4: Evaluate ingredient extraction on all records

total_correct = 0

true_positives = 0
false_positives = 0
false_negatives = 0

for _, row in df.iterrows():

    expected = set(row["expected_names"])

    detected = set(extract_ingredients(row["ocr_text"]))

    # Exact match
    if expected == detected:
        total_correct += 1

    # Calculate TP, FP and FN
    true_positives += len(expected & detected)
    false_positives += len(detected - expected)
    false_negatives += len(expected - detected)


# Calculate evaluation metrics
accuracy = (total_correct / len(df)) * 100

precision = (
    true_positives / (true_positives + false_positives)
    if true_positives + false_positives > 0 else 0
)

recall = (
    true_positives / (true_positives + false_negatives)
    if true_positives + false_negatives > 0 else 0
)

f1_score = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0 else 0
)

print("INGREDIENT EXTRACTION EVALUATION")
print("--------------------------------")
print("Total Records:", len(df))
print("Exact Matches:", total_correct)
print("Exact Match Accuracy:", round(accuracy, 2), "%")
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1_score, 4))


INGREDIENT EXTRACTION EVALUATION
--------------------------------
Total Records: 500
Exact Matches: 500
Exact Match Accuracy: 100.0 %
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


### 6. Identifying Incorrect Ingredient Predictions

In [8]:

# Find records where detected ingredients differ from expected

print("INCORRECTLY MATCHED RECORDS")
print("--------------------------------")

for _, row in df.iterrows():

    expected = set(row["expected_names"])
    detected = set(extract_ingredients(row["ocr_text"]))

    if expected != detected:

        print("\nMedicine ID:", row["id"])
        print("Expected:", expected)
        print("Detected:", detected)
        print("Extra:", detected - expected)
        print("Missing:", expected - detected)


INCORRECTLY MATCHED RECORDS
--------------------------------
